# 03 — Model Training & Evaluation

Trains the Random Forest baseline and the multiclass XGBoost model, evaluates both, and saves the final model + label encoder. **The XGBoost cell is the slow one** — budget real time for this on CPU (this took ~80 minutes at n_estimators=300 during development). Smoke-test with a small n_estimators first if you've changed anything upstream.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

X = np.load('../data/X_fingerprints.npy')
y = np.load('../data/y_labels.npy', allow_pickle=True)
print(X.shape, y.shape)


In [ ]:
# Drop rows with invalid SMILES (either half of the pair all-zero)
valid_mask = ~((X[:, :2048].sum(axis=1) == 0) | (X[:, 2048:].sum(axis=1) == 0))
print(f"Dropping {(~valid_mask).sum()} rows ({(~valid_mask).sum()/len(X):.2%})")

X_valid = X[valid_mask]
y_valid = y[valid_mask].astype(str)  # avoids int/str sort crash in sklearn


In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_valid)
print(f"Classes: {len(le.classes_)}")


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_valid, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)


## Random Forest baseline

In [ ]:
from src.model import train_random_forest, evaluate_model

rf = train_random_forest(X_train, y_train)
rf_metrics = evaluate_model(rf, X_val, y_val, le)


## XGBoost (final model)

In [ ]:
from src.model import train_xgboost

model = train_xgboost(X_train, y_train, X_val, y_val, num_class=len(le.classes_))
test_metrics = evaluate_model(model, X_test, y_test, le)


## Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = test_metrics['y_pred']

cm = confusion_matrix(y_test, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(20, 20))
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, xticks_rotation=90, values_format='.2f', colorbar=True)
plt.tight_layout()
plt.savefig('../models/confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
N = 15
top_classes_idx = np.argsort(-np.bincount(y_test, minlength=len(le.classes_)))[:N]
top_class_names = le.classes_[top_classes_idx]

mask = np.isin(y_test, top_classes_idx)
y_test_top = y_test[mask]
y_pred_top = y_pred[mask]

cm_top = confusion_matrix(y_test_top, y_pred_top, labels=top_classes_idx, normalize='true')

fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(cm_top, display_labels=top_class_names)
disp.plot(ax=ax, xticks_rotation=90, values_format='.2f', colorbar=True, cmap='Blues')
plt.title(f'Confusion Matrix — Top {N} Most Frequent Classes')
plt.tight_layout()
plt.savefig('../models/confusion_matrix_top15.png', dpi=150)
plt.show()


## Save the model

In [ ]:
import joblib

joblib.dump(model, '../models/ddi_xgb.joblib')
joblib.dump(le, '../models/label_encoder.joblib')
print('Saved.')
